# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone https://github.com/likithagarlapati7-oss/flyrank-ml-internship
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 132 (delta 42), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 1.87 MiB | 8.60 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/flyrank-ml-internship


In [3]:
import pandas as pd

feature_df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

feature_df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [4]:
feature_df["refresh_needed"] = (
    (feature_df["ctr"] < feature_df["ctr"].median()) &
    (feature_df["days_since_last_update"] > feature_df["days_since_last_update"].median())
).astype(int)

In [5]:
features = [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "search_volume",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
    "trend_pct"
]

X = feature_df[features]
y = feature_df["refresh_needed"]

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# Finding 1: AI-generated pages received higher impressions and clicks than human-only pages

# The paper reports that AI-assisted content generally achieved higher search impressions and clicks than human-only content based on aggregate comparisons across a large number of pages. The analysis compares observed search performance metrics between different content creation approaches rather than claiming that AI directly causes better SEO performance.

# Methodology Question

# Where does the label come from?

# The outcome measures (impressions and clicks) come from observed historical search performance data, making them real measured outcomes rather than manually assigned labels.

# Does the validation design support the claim?

# The aggregate comparison supports the descriptive claim that AI-assisted pages were associated with higher search performance in the observed dataset. However, because this is an observational comparison, it does not establish that AI alone caused the improvement. Other factors such as topic selection, client characteristics, or publishing strategy may also influence the results. The claim is therefore best interpreted as an observed association rather than causal evidence

In [ ]:
# Finding 2: The machine learning model can distinguish AI-generated from human-written content

# The paper presents a machine learning model that predicts whether content was AI-generated or human-written using content features. The authors describe these results as exploratory and include them in the appendix rather than using them as the primary evidence for their conclusions.

# Methodology Question

# Where does the label come from?

# The labels are based on the known content production process (AI-generated versus human-written), providing a clearly defined supervised learning target.

# Does the validation design support the claim?

# The validation demonstrates that the model can distinguish between the two content types within the available dataset. However, additional information about how the train-test split prevents overlap between similar clients, topics, or content would further strengthen confidence that the model generalizes beyond the observed data. Treating these results as exploratory, rather than definitive proof, is an appropriate methodological choice.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
## 2. My Model Under an Honest Split (Before/After)

# In Week 5, I evaluated the model using a random 80:20 train-test split. While this provides an estimate of performance, pages from the same client may appear in both the training and test sets. This can make the evaluation optimistic because client-specific patterns may already be present during training.

# For this validation audit, I re-evaluated the model using a **grouped split based on `client_id`**. This ensures that all pages from a single client appear in only one split, providing a more realistic estimate of how well the model generalizes to unseen clients.

# This grouped split is more appropriate because the real-world task is to score pages for clients that the model has not previously learned from.

In [6]:
from sklearn.model_selection import GroupShuffleSplit

groups = feature_df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [7]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [8]:
feature_df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct',
 'refresh_needed']

In [10]:
from sklearn.model_selection import GroupShuffleSplit

groups = feature_df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print(X_train.shape)
print(X_test.shape)


(23837, 12)
(6163, 12)


In [11]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [12]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5101
           1       1.00      1.00      1.00      1062

    accuracy                           1.00      6163
   macro avg       1.00      1.00      1.00      6163
weighted avg       1.00      1.00      1.00      6163



In [ ]:
# ### Interpretation

# In Week 5, the model was evaluated using a random 80:20 train-test split. For this validation audit, I re-ran the model using a grouped split based on `client_id`, ensuring that pages from the same client did not appear in both the training and test sets.

# The evaluation metrics remained unchanged after applying the grouped split. While this suggests that the model performs consistently across different clients, these results should be interpreted carefully.

# The proxy target (`refresh_needed`) was created using two variables: `ctr` and `days_since_last_update`. These same variables were also included as input features. As a result, the Random Forest model is primarily learning the rule used to construct the proxy target rather than predicting independently observed refresh outcomes.

# Therefore, the perfect evaluation scores demonstrate that the model reproduces the proxy rule accurately. They should not be interpreted as evidence that the model can predict which pages will truly benefit from a content refresh. The model is best viewed as a decision-support tool based on historical signals rather than proof of future performance.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [14]:
## 3. Leakage Audit

# The final model uses only historical information that would be available before deciding whether to refresh a content page. No future performance metrics, post-refresh outcomes, or internal FlyRank recommendation flags were included as input features.

# The feature audit is shown below.

# | Feature | Available at Decision Time? | Leakage Risk | Notes |
# |---------|-----------------------------|--------------|-------|
# | impressions_90d | Yes | Low | Historical search visibility |
# | clicks_90d | Yes | Low | Historical click data |
# | pageviews_90d | Yes | Low | Historical traffic metric |
# | sessions_90d | Yes | Low | Historical engagement |
# | engaged_sessions_90d | Yes | Low | Historical engagement |
# | search_volume | Yes | Low | Topic demand known before decision |
# | avg_position | Yes | Low | Historical search ranking |
# | engagement_rate | Yes | Low | Computed from historical sessions |
# | content_age_days | Yes | Low | Existing page metadata |
# | trend_pct | Yes | Low | Historical performance trend |
# | ctr | Yes | Moderate | Used to construct the proxy target |
# | days_since_last_update | Yes | Moderate | Used to construct the proxy target |

# ### Leakage Assessment

# No future information or post-refresh outcomes were used during training. The model only uses information that would be available before making a refresh decision.

# However, the proxy label (`refresh_needed`) was created directly from `ctr` and `days_since_last_update`. These same variables were also included as model features. This does not represent future-data leakage, but it does make the learning task easier because the model can learn the rule used to generate the proxy labels.

# As a result, the perfect evaluation metrics should be interpreted as the model successfully reproducing the proxy rule rather than demonstrating that it can predict true refresh outcomes.

# Therefore, the model should be viewed as a decision-support system based on historical signals rather than as proof that a page will benefit from a refresh.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
## 4. Claim Rewrite

### Original Claim

# "The Random Forest model identifies pages that require a content refresh."

# ### Revised Claim

# Based on the historical performance metrics available in this dataset, the Random Forest model provides a **directional ranking** of pages that may benefit from further review. The model was evaluated using a proxy target derived from historical signals, so the results should be interpreted as **observed** and **measured** patterns rather than proof that a page truly requires a refresh. The output is intended as **decision-support** for the content team and should be combined with human review before taking action.

# ### Why the Claim Was Rewritten

# The evaluation shows that the model accurately reproduces the proxy rule used to create the training labels. However, because the proxy target is not a direct record of successful content refreshes, the model cannot establish causal effects or guarantee future improvements after a page is updated.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.